In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tmrl.envs import GenericGymEnv
from tmrl.config.config_objects import CONFIG_DICT
import torch.distributions as dist

In [2]:
env = GenericGymEnv(id="real-time-gym-ts-v1", gym_kwargs={"config": CONFIG_DICT})

c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\rtgym\envs\real_time_env.py:662: UserWarning: Time-step timed out. Elapsed since last time-step: 0.10984639999878709
  warnings.warn(f"Time-step timed out. Elapsed since last time-step: {now - self.__t_end}")
c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\rtgym\envs\real_time_env.py:662: UserWarning: Time-step timed out. Elapsed since last time-step: 0.32248169999911624
  warnings.warn(f"Time-step timed out. Elapsed since last time-step: {now - self.__t_end}")
c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\rtgym\envs\real_time_env.py:662: UserWarning: Time-step timed out. Elapsed since last time-step: 0.0633986000002551
  warnings.warn(f"Time-step timed out. Elapsed since last time-step: {now - self.__t_end}")
c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\rtgym\envs\real_time_env.py:662: UserWarning: Time-step timed out. Elapsed since l

In [3]:
TAMANHO_INPUT = 83 # 1 (vel) + 19 (lidar) + 6 (histórico de ações)
TAMANHO_OUTPUT = 3 # Acelerar, Frear, Virar

In [4]:
class Ator(nn.Module):
    def __init__(self):
        super(Ator, self).__init__()
        self.rede = nn.Sequential(
            nn.Linear(TAMANHO_INPUT, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, TAMANHO_OUTPUT),
            nn.Tanh()
        )

    def forward(self, x):
        return self.rede(x)

In [5]:
class Critico(nn.Module):
    def __init__(self):
        super(Critico, self).__init__()
        self.rede = nn.Sequential(
            nn.Linear(TAMANHO_INPUT, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.rede(x)

In [6]:
ator = Ator()
critico = Critico()

In [7]:
otimizador_ator = optim.Adam(ator.parameters(), lr=0.0001)
otimizador_critico = optim.Adam(critico.parameters(), lr=0.0005)

In [8]:
print("Dando o play na pista... CLIQUE NA JANELA DO JOGO AGORA!")
obs, info = env.reset()

for passo in range(1000):
    # 1. PREPARAÇÃO DO ESTADO ATUAL
    velocidade = np.array(obs[0]).flatten()
    lidar = np.array(obs[1]).flatten()
    acoes = [np.array(a).flatten() for a in obs[2:]]
    historico_acoes = np.concatenate(acoes) if len(acoes) > 0 else np.array([])
    
    estado_atual = np.concatenate([velocidade, lidar, historico_acoes])
    estado_tensor = torch.tensor(estado_atual, dtype=torch.float32)
    
    # 2. O ATOR DECIDE A AÇÃO (Com Exploração)
    mu = ator(estado_tensor)
    
    # Criamos um desvio padrão fixo (0.1) para ele explorar o ambiente
    std = torch.ones_like(mu) * 0.4 
    distribuicao = dist.Normal(mu, std)
    
    # Sorteia a ação baseada na distribuição e limita entre -1 e 1
    acao = distribuicao.sample()
    acao = torch.clamp(acao, -1.0, 1.0)
    
    # Precisamos da probabilidade (log_prob) dessa ação para a matemática do Ator
    log_prob = distribuicao.log_prob(acao).sum()
    
    # O Crítico avalia o quão bom é o estado ATUAL
    valor_atual = critico(estado_tensor)
    
    # 3. INTERAÇÃO COM O TRACKMANIA
    acao_numpy = acao.detach().numpy()
    proximo_obs, recompensa_crua, terminated, truncated, info = env.step(acao_numpy)
    done = terminated or truncated

    recompensa = recompensa_crua
    velocidade_kmh = proximo_obs[0][0]

    if velocidade_kmh < 5.0:
        recompensa -= 0.5

    print(f"AÇÃO: [Acelerar/Frear/Virar]: {acao_numpy} | VEL: {velocidade_kmh:.1f} | RECOMP: {recompensa}")

    # 4. PREPARAÇÃO DO PRÓXIMO ESTADO
    prox_velocidade = np.array(proximo_obs[0]).flatten()
    prox_lidar = np.array(proximo_obs[1]).flatten()
    prox_acoes = [np.array(a).flatten() for a in proximo_obs[2:]]
    prox_historico = np.concatenate(prox_acoes) if len(prox_acoes) > 0 else np.array([])
    
    proximo_estado = np.concatenate([prox_velocidade, prox_lidar, prox_historico])
    proximo_estado_tensor = torch.tensor(proximo_estado, dtype=torch.float32)
    
    # O Crítico avalia o NOVO estado (sem calcular gradiente para não bugar o grafo)
    with torch.no_grad():
        valor_proximo = critico(proximo_estado_tensor)
        
    # 5. A MATEMÁTICA DO ACTOR-CRITIC
    gamma = 0.99
    
    # Advantage: Recompensa recebida + Estimativa Futura - Estimativa Original
    # O (1 - int(done)) garante que não somaremos valor futuro se a corrida acabou (bateu/venceu)
    advantage = recompensa + (gamma * valor_proximo * (1 - int(done))) - valor_atual
    
    # Loss do Crítico: Erro quadrático simples. Ele quer prever o Advantage com perfeição.
    loss_critico = advantage.pow(2)
    
    print("LOSS CRÍTICO: " + str(loss_critico))

    # Loss do Ator: Se o Advantage foi positivo, o log_prob é incentivado. Se negativo, punido.
    # O .detach() é vital: ele impede que o otimizador do Ator altere os pesos do Crítico.
    loss_ator = -log_prob * advantage.detach()
    
    print("LOSS ATOR: " + str(loss_ator))

    # 6. BACKPROPAGATION (Atualizando os Cérebros)
    otimizador_ator.zero_grad()
    loss_ator.backward()
    otimizador_ator.step()
    
    otimizador_critico.zero_grad()
    loss_critico.backward()
    otimizador_critico.step()
    
    # Prepara o loop para o próximo frame
    if done:
        obs, info = env.reset()
    else:
        obs = proximo_obs

print("Treinamento local concluído!")

Dando o play na pista... CLIQUE NA JANELA DO JOGO AGORA!
AÇÃO: [Acelerar/Frear/Virar]: [ 0.2563052  0.082554  -1.       ] | VEL: 0.0 | RECOMP: -0.5
LOSS CRÍTICO: tensor([0.2606], grad_fn=<PowBackward0>)
LOSS ATOR: tensor([-0.9566], grad_fn=<MulBackward0>)
AÇÃO: [Acelerar/Frear/Virar]: [-0.31328028  0.6965594  -0.25068694] | VEL: 0.0 | RECOMP: -0.5
LOSS CRÍTICO: tensor([0.2529], grad_fn=<PowBackward0>)
LOSS ATOR: tensor([-0.7451], grad_fn=<MulBackward0>)
AÇÃO: [Acelerar/Frear/Virar]: [-0.12408727 -0.04533803 -0.16685304] | VEL: 0.3 | RECOMP: -0.5
LOSS CRÍTICO: tensor([0.2509], grad_fn=<PowBackward0>)
LOSS ATOR: tensor([-0.4239], grad_fn=<MulBackward0>)
AÇÃO: [Acelerar/Frear/Virar]: [-0.02123803  0.75823927 -0.672966  ] | VEL: 0.8 | RECOMP: -0.5
LOSS CRÍTICO: tensor([0.2307], grad_fn=<PowBackward0>)
LOSS ATOR: tensor([-0.8269], grad_fn=<MulBackward0>)
AÇÃO: [Acelerar/Frear/Virar]: [ 0.6264647  -0.00737885  0.01024365] | VEL: 0.9 | RECOMP: -0.5
LOSS CRÍTICO: tensor([0.2627], grad_fn=<PowB